In [2]:
import os
import sys
# Add the parent directory to the Python path to allow module imports
sys.path.append(os.path.dirname(os.getcwd()))

import matplotlib
import matplotlib.pyplot as plt
# Use the "Agg" backend for matplotlib to avoid issues
matplotlib.use("Agg")

import json
import librosa
import numpy as np
import pandas as pd
import shutil
import soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image


from modules.dataset import ICBHIAudioDataset, KAUHAudioDataset
from modules.lungsound import LungSoundAudio
from modules.transforms import *

In [3]:
DATA_PATH = Path(os.path.join(os.path.dirname(os.getcwd()), "data"))
RAW_DATA_FOLDER = DATA_PATH / "raw"
INTERIM_DATA_FOLDER = DATA_PATH / "interim"
PREPROCESSED_DATA_FOLDER = DATA_PATH / "preprocessed"

if not os.path.exists(RAW_DATA_FOLDER):
    raise FileNotFoundError(f"Raw data folder not found at {RAW_DATA_FOLDER}. Please ensure the original data was already downloaded and placed in the correct location.")

if not os.path.exists(INTERIM_DATA_FOLDER):
    os.makedirs(INTERIM_DATA_FOLDER)
    print(f"Created interim data folder at {INTERIM_DATA_FOLDER}.")
else:
    if len(os.listdir(INTERIM_DATA_FOLDER)) > 0:
        print(f"[WARNING] Interim data folder already exist and is not empty ({INTERIM_DATA_FOLDER}). Consider deleting it to run the preprocessing step again.")

if not os.path.exists(PREPROCESSED_DATA_FOLDER):
    os.makedirs(PREPROCESSED_DATA_FOLDER)
    print(f"Created preprocessed data folder at {PREPROCESSED_DATA_FOLDER}.")
else:
    if len(os.listdir(PREPROCESSED_DATA_FOLDER)) > 0:
        print(f"[WARNING] Preprocessed data folder already exist and is not empty ({PREPROCESSED_DATA_FOLDER}). Consider deleting it to run the preprocessing step again.")

[WARNING] Interim data folder already exist and is not empty (/home/leticialopes/Projects/IA901/IA901_Project/data/interim). Consider deleting it to run the preprocessing step again.
Created preprocessed data folder at /home/leticialopes/Projects/IA901/IA901_Project/data/preprocessed.


## Interim

In [ ]:
TARGET_SR = 22050   # Hz
WINDOW_LENGTH = 5.0 # seconds
HOP_LENGTH = 5.0    # seconds

def preprocess_audios(original_data_path: Path, preprocessed_data_path: Path):
    """
    Preprocesses the original data and saves the preprocessed data to the specified location.
    Args:
        original_data_path (Path): Path to the original raw data.
        preprocessed_data_path (Path): Path where the preprocessed data will be saved.
    """
    # Grab all files in the original data directory, .wav or not, including subdirectories
    all_files = sorted(original_data_path.glob("**/*.*"))
    # Iterate through all files and apply preprocessing to audio files, while copying non-audio files
    wav_count = 0
    for file in tqdm(all_files, desc=f"Preprocessing {original_data_path.name}"):
        if file.suffix.lower() == ".wav":
            wav_count += 1
            # Load the audio file using the LungSound class
            audio = LungSoundAudio(str(file))
            # Apply preprocessing transforms
            # 1. Split the audio into windows of fixed duration
            cropped_audios = Window(window_length=WINDOW_LENGTH, hop_length=HOP_LENGTH)(audio)
            for i, cropped_audio in enumerate(cropped_audios):
                # 2. Resample the audio to the target sampling rate
                resampled_audio = Resample(target_sr=TARGET_SR)(cropped_audio)
                # 3. Normalize the audio to have zero mean and unit variance
                normalized_audio = NormalizeAudio()(resampled_audio)
                # Save the preprocessed audio to the new location
                start = int(i * HOP_LENGTH)
                end = int(start + WINDOW_LENGTH)
                new_file_name = f"{file.stem}_clip-{start:03d}-{end:03d}.wav"
                relative_path = file.parent.relative_to(original_data_path)
                preprocessed_file_path = preprocessed_data_path / relative_path / new_file_name
                preprocessed_file_path.parent.mkdir(parents=True, exist_ok=True)
                sf.write(preprocessed_file_path, normalized_audio.audio, normalized_audio.sr)

        else:
            # If it's not an audio file, simply copy it to the new location
            relative_path = file.parent.relative_to(original_data_path)
            new_file_path = preprocessed_data_path / relative_path / file.name
            new_file_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(file, new_file_path)

    print(f"Preprocessing completed for '{original_data_path.name}'.")
    print(f"Total audio files processed: {wav_count}")

    # Save a file containing the total number of preprocessed audio files
    audio_transforms = {
        Window.__name__: {"params": vars(Window(window_length=WINDOW_LENGTH, hop_length=HOP_LENGTH))},
        Resample.__name__: {"params": vars(Resample(target_sr=TARGET_SR))},
        NormalizeAudio.__name__: {"params": vars(NormalizeAudio())},
    }
    with open(preprocessed_data_path / "preprocessing.json", "w") as f:
        json.dump({"audio_transforms": audio_transforms}, f, indent=4)

In [ ]:
preprocess_audios(RAW_DATA_FOLDER, INTERIM_DATA_FOLDER)

Preprocessing raw:   0%|          | 0/2194 [00:00<?, ?it/s]

Preprocessing completed for raw.
Total audio files processed: 1256


## Preprocessed

In [ ]:
def save_features_as_npy(features: np.ndarray, path: str):
    """
    Saves the extracted features to a .npy file.
    Args:
        features (np.ndarray): The extracted features to be saved.
        path (str): The path where the features will be saved.
    """
    np.save(path, features)

def save_features_as_npz(features: np.ndarray, sr: int, path: str):
    """
    Saves the extracted features and sampling rate to a .npz file.
    Args:
        features (np.ndarray): The extracted features to be saved.
        sr (int): The sampling rate associated with the features.
        path (str): The path where the features will be saved.
    """
    np.savez(path, features=features, sr=sr)


def save_features_as_png(features: np.ndarray, path: str, spec_params: dict = None):
    """
    Saves the extracted features as a PNG image.
    Args:
        features (np.ndarray): The extracted features to be saved.
        path (str): The path where the features will be saved.
        spec_params (dict): Parameters for the spectrogram visualization.
    """
    if features.ndim == 3 and features.shape[-1] == 3:
        fig, ax = plt.subplots()
        ax.imshow(features)
        ax.axis("off")
        fig.savefig(path, bbox_inches="tight", pad_inches=0)
        ax.clear()
        fig.clf()
        plt.close(fig)
    # If the features are 2D (e.g., spectrogram), use librosa's specshow to visualize and save them as an image
    elif features.ndim == 2:
        params = {} if spec_params is None else spec_params
        fig, ax = plt.subplots()
        librosa.display.specshow(features, ax=ax, **params)
        ax.axis("off")
        fig.savefig(path, bbox_inches="tight", pad_inches=0)
        ax.clear()
        fig.clf()
        plt.close(fig)
    else:
        raise ValueError(f"Unsupported feature shape for PNG saving: {features.shape}")


def preprocess_features(original_data_path: Path, preprocessed_data_path: Path, save_as: str = "npy"):
    """
    Preprocesses the original data and saves the preprocessed data to the specified location using multiple feature extractors.
    Args:
        original_data_path (Path): Path to the original raw data.
        preprocessed_data_path (Path): Path where the preprocessed data will be saved.
        save_as (str): Format to save the preprocessed data ('npy', 'npz', or 'png').
    """
    datasets = [
        ICBHIAudioDataset(original_data_path),
        KAUHAudioDataset(original_data_path)
    ]

    feature_extractors = {
        "STFT": STFT(),
        "MelSpectrogram": MelSpectrogram(n_mels=128),
        "MFCC": MFCC(n_mfcc=128),
        "MFCCDelta": MFCCDelta(n_mfcc=128),
        "Chroma": Chroma(n_chroma=128),
        "SpectralContrast": SpectralContrast(n_bands=4, fmin=50),
        "CQT": CQT(n_bins=48, fmin=30, bins_per_octave=12),
        "Phase": Phase(),
        "Stack_MelSpectrogram_MFCC_Chroma": Stack([
            MelSpectrogram(n_mels=128),
            MFCC(n_mfcc=128),
            Chroma(n_chroma=128),
        ])
    }

    # Iterate through each dataset and apply preprocessing with each feature extractor
    for dataset in datasets:
        df = dataset.data
        new_rows = []
        computed_data = False
        for feature_extractor_name, feature_extractor in feature_extractors.items():
            for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Preprocessing {dataset.name} with {feature_extractor.name}"):
                audio_path = Path(row["FilePath"])
                # Load the audio file using the LungSound class
                audio = LungSoundAudio(audio_path)
                # Extract features using the provided feature extractor
                features = feature_extractor(audio)
                # Normalize the features to mantain the values between 0 and 1
                # features = NormalizeFeatures()(features)

                # Save the preprocessed audio to the new location
                diagnosis = str(row["Diagnosis"])
                new_file_name = f"{audio_path.stem}.{save_as}"
                preprocessed_file_path = preprocessed_data_path / dataset.name / feature_extractor_name / diagnosis / new_file_name
                preprocessed_file_path.parent.mkdir(parents=True, exist_ok=True)

                if save_as == "npy":
                    save_features_as_npy(features, preprocessed_file_path)
                elif save_as == "npz":
                    save_features_as_npz(features, audio.sr, preprocessed_file_path)
                elif save_as == "png":
                    params = feature_extractor.plot_params
                    params["sr"] = TARGET_SR
                    save_features_as_png(features, preprocessed_file_path, params)
                else:
                    raise ValueError(f"Unsupported save format: {save_as}")

                # Add the row to the new df
                if not computed_data:
                    new_row = row.copy()
                    new_row["FilePath"] = new_file_name
                    new_rows.append(new_row)

            computed_data = True
            # Save the preprocessing parameters to a json file
            preprocessing_path = preprocessed_data_path / dataset.name / feature_extractor.name / "preprocessing.json"
            feature_extractor = {
                feature_extractor.__class__.__name__: {
                    "params": vars(feature_extractor),
                    "plot_params": feature_extractor.plot_params
                }
            }
            feature_transforms = {
                NormalizeFeatures.__name__: {"params": vars(NormalizeFeatures())}
            }
            preprocessing = {
                "feature_extractor": feature_extractor,
                # "feature_transforms": feature_transforms
            }
            with open(preprocessing_path, "w") as f:
                json.dump(preprocessing, f, indent=4, default=str)

        # Save the new data to a CSV file
        data_path = preprocessed_data_path / dataset.name / "data.csv"
        new_df = pd.DataFrame(new_rows).rename(columns={"FilePath": "FileName"})
        new_df.to_csv(data_path, index=False)

        print(f"Preprocessing for {dataset.name} completed. Total preprocessed audio files: {len(new_rows)}")
        print(f"Data saved to {os.path.relpath(data_path, start=os.getcwd())}")

In [6]:
preprocess_features(INTERIM_DATA_FOLDER, PREPROCESSED_DATA_FOLDER, save_as="npz")

Preprocessing ICBHI with STFT:   0%|          | 0/3901 [00:00<?, ?it/s]

Preprocessing ICBHI with MelSpectrogram:   0%|          | 0/3901 [00:00<?, ?it/s]

Preprocessing ICBHI with MFCC:   0%|          | 0/3901 [00:00<?, ?it/s]

Preprocessing ICBHI with MFCCDelta:   0%|          | 0/3901 [00:00<?, ?it/s]

Preprocessing ICBHI with Chroma:   0%|          | 0/3901 [00:00<?, ?it/s]

...

Preprocessing KAUH with CQT:   0%|          | 0/940 [00:00<?, ?it/s]

Preprocessing KAUH with Phase:   0%|          | 0/940 [00:00<?, ?it/s]

Preprocessing KAUH with Stack_MelSpectrogram_MFCC_Chroma:   0%|          | 0/940 [00:00<?, ?it/s]

/home/leticialopes/miniconda3/envs/lung_sounds/lib/python3.13/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


Preprocessing for KAUH completed. Total preprocessed audio files: 940
Data saved to ../data/preprocessed/KAUH/data.csv
